## NUM_EV

In [ ]:
import pandas as pd
import os
import sys
from tabulate import tabulate

def leer_archivo(ruta_archivo: str):
    # Verificar que el archivo existe
    if not os.path.exists(ruta_archivo):
        sys.exit(f"❌ El archivo no existe: {ruta_archivo}")

    # Detectar la extensión
    extension = os.path.splitext(ruta_archivo)[1].lower()

    try:
        if extension == ".csv":
            df = pd.read_csv(
                ruta_archivo,
                thousands=",",         # ✅ interpreta comas como miles (2022, 20000)
                na_values=["", " "],   # ✅ vacíos como NaN
                keep_default_na=True
            )
        elif extension in [".xls", ".xlsx"]:
            df = pd.read_excel(ruta_archivo)
        elif extension == ".json":
            df = pd.read_json(ruta_archivo)
        else:
            sys.exit(f"⚠️ Tipo de archivo no soportado: {extension}")

        # Llamada a la función para mostrar los primeros 10 datos
        mostrar_datos(df, 10)
        return df

    except Exception as e:
        sys.exit(f"❌ Error al leer el archivo: {e}")

def mostrar_datos(df, num_filas=10):
    """
    Muestra las primeras filas del DataFrame de una manera bonita.
    Por defecto, muestra las primeras 10 filas, pero se puede especificar un número diferente.
    
    :param df: El DataFrame a mostrar
    :param num_filas: El número de filas a mostrar (por defecto 10)
    """
    print(f"\n🔍 Mostrando las primeras {num_filas} filas del DataFrame:\n")
    
    # Mostrar las primeras 'num_filas' filas de manera bonita con tabulate
    print(tabulate(df.head(num_filas), headers='keys', tablefmt='pretty', showindex=False))

In [ ]:
import matplotlib.pyplot as plt

# Ruta del archivo
ruta = "./data/num_ev.csv"
df = leer_archivo(ruta)
df.head(10)

In [ ]:
# ---------------------------
# Limpieza de columnas numéricas
# ---------------------------
# Asegurar que algunas columnas sean numéricas
cols_numericas = ["AÑO_REGISTRO", "CAPACIDAD_PASAJEROS", "POTENCIA"]
for col in cols_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.drop(columns=["COMBUSTIBLE", "ESTADO", "MODELO", "FECHA_REGISTRO", "CLASIFICACION", "LINEA", "CARROCERIA", "CILINDRAJE", "MODALIDAD", "ORGANISMO_TRANSITO", "CAPACIDAD_CARGA", "CAPACIDAD_PASAJEROS", "PESO", "EJES"])

In [ ]:
# 📈 Gráfica dinámica: Evolución de vehículos eléctricos registrados por año
import plotly.express as px

if df is not None and not df.empty:
    # Agrupar por año y ordenar cronológicamente
    conteo_anual = (
        df["AÑO_REGISTRO"]
        .value_counts()
        .reset_index()
    )
    conteo_anual.columns = ["Año", "Cantidad"]
    conteo_anual = conteo_anual.sort_values("Año")  # ✅ Corrección clave

    print("📈 Creando gráfica de vehículos registrados por año...")

    # Crear gráfico dinámico
    fig_vehiculos_anual = px.line(
        conteo_anual,
        x="Año",
        y="Cantidad",
        text="Cantidad",  # ✅ Mostrar valores encima de los puntos
        markers=True,
        title="📈 Evolución de Vehículos Eléctricos Registrados por Año",
        labels={"Año": "Año de registro", "Cantidad": "Cantidad de vehículos"},
    )

    # Personalización de estilo
    fig_vehiculos_anual.update_traces(
        line=dict(color="royalblue", width=3),
        marker=dict(size=9, line=dict(color="black", width=1)),
        textposition="top center"
    )

    # Ajustes visuales
    fig_vehiculos_anual.update_layout(
        height=500,
        xaxis_tickangle=-45,
        plot_bgcolor="white",
        xaxis=dict(showgrid=True, gridcolor="lightgray"),
        yaxis=dict(showgrid=True, gridcolor="lightgray"),
    )

    fig_vehiculos_anual.show()

else:
    print("❌ No hay datos disponibles para crear la gráfica de vehículos por año")


In [ ]:
# 🚗 Gráfica dinámica: Top 10 marcas de vehículos eléctricos
import plotly.express as px

if df is not None and not df.empty:
    # Obtener las 10 marcas más comunes
    top_marcas = (
        df["MARCA"]
        .value_counts()
        .head(10)
        .reset_index()
    )
    top_marcas.columns = ["Marca", "Cantidad"]

    print("🚗 Creando gráfica de Top 10 marcas de vehículos eléctricos...")

    # Crear gráfico de barras horizontal
    fig_marcas = px.bar(
        top_marcas,
        x="Cantidad",
        y="Marca",
        orientation="h",
        text="Cantidad",  # ✅ Etiquetas con valores
        title="🚗 Top 10 Marcas de Vehículos Eléctricos",
        labels={"Cantidad": "Cantidad de vehículos", "Marca": "Marca"},
        color="Marca",  # ✅ Colores diferenciados
        color_discrete_sequence=px.colors.qualitative.Vivid
    )

    # Personalización del estilo
    fig_marcas.update_traces(
        textposition="outside",
        marker=dict(line=dict(color="black", width=1))  # ✅ Borde negro como las anteriores
    )

    # Ajustes de diseño
    fig_marcas.update_layout(
        height=500,
        xaxis_title="Cantidad de vehículos",
        yaxis_title="Marca",
        yaxis=dict(categoryorder="total ascending"),  # ✅ Orden ascendente (como en seaborn)
        plot_bgcolor="white",
        showlegend=False
    )

    fig_marcas.show()

else:
    print("❌ No hay datos disponibles para crear la gráfica de marcas")


In [ ]:
# 🛻 Gráfica dinámica: Distribución por clase de vehículo
import plotly.express as px

if df is not None and not df.empty:
    # ✅ Calcular distribución de clases
    class_counts = df["CLASE"].value_counts()

    # ✅ Agrupar clases menores al 1% en "Otros"
    threshold = class_counts.sum() * 0.01
    small_classes = class_counts[class_counts < threshold].index
    df["CLASE"] = df["CLASE"].replace(small_classes, "Otros")

    # ✅ Recalcular distribución
    class_counts = df["CLASE"].value_counts().reset_index()
    class_counts.columns = ["Clase", "Cantidad"]

    print("🛻 Creando gráfica de distribución por clase de vehículo...")

    # ✅ Crear gráfico circular con Plotly
    fig_clase = px.pie(
        class_counts,
        names="Clase",
        values="Cantidad",
        title="🛻 Distribución por Clase de Vehículo",
        color="Clase",
        color_discrete_sequence=px.colors.qualitative.Set2,  # 🎨 Colores suaves y agradables
        hole=0.3  # 🔘 estilo donut (más moderno)
    )

    # ✅ Ajustes visuales
    fig_clase.update_traces(
        textposition="inside",
        textinfo="percent+label",  # muestra porcentaje y nombre
        marker=dict(line=dict(color="black", width=1))  # borde negro estilo uniforme
    )

    fig_clase.update_layout(
        height=500,
        showlegend=True,
        legend_title_text="Clase de Vehículo",
        plot_bgcolor="white"
    )

    fig_clase.show()

else:
    print("❌ No hay datos disponibles para crear la gráfica de clases de vehículo")


In [ ]:
# ⚙️ Gráfica dinámica: Potencia promedio por marca (Top 10)
import plotly.express as px

if df is not None and not df.empty and "POTENCIA" in df.columns:
    # ✅ Calcular promedio de potencia por marca
    potencia_marca = (
        df.groupby("MARCA")["POTENCIA"]
        .mean()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )

    print("⚙️ Creando gráfica de potencia promedio por marca...")

    # ✅ Crear gráfica de barras dinámica
    fig_potencia = px.bar(
        potencia_marca,
        x="MARCA",
        y="POTENCIA",
        title="⚙️ Potencia Promedio por Marca (Top 10)",
        labels={"MARCA": "Marca", "POTENCIA": "Potencia Promedio (HP o kW)"},
        text="POTENCIA",
        color="MARCA",
        color_discrete_sequence=px.colors.qualitative.Pastel1
    )

    # ✅ Personalización visual
    fig_potencia.update_traces(
        texttemplate="%{text:.2f}",  # muestra potencia con dos decimales
        textposition="outside",
        marker=dict(line=dict(color="black", width=1))
    )

    fig_potencia.update_layout(
        height=500,
        xaxis_tickangle=-45,
        showlegend=False,
        plot_bgcolor="white"
    )

    fig_potencia.show()

else:
    print("❌ No hay datos disponibles o la columna 'POTENCIA' no existe para crear la gráfica")


In [ ]:
# 🧍‍♂️ Gráfica dinámica: Capacidad promedio de pasajeros por tipo de servicio
import plotly.express as px

if (
    df is not None 
    and not df.empty 
    and "CAPACIDAD_PASAJEROS" in df.columns 
    and "SERVICIO" in df.columns
):
    # ✅ Calcular capacidad promedio de pasajeros por servicio
    capacidad_servicio = (
        df.groupby("SERVICIO")["CAPACIDAD_PASAJEROS"]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )

    print("🧍‍♂️ Creando gráfica de capacidad promedio de pasajeros por servicio...")

    # ✅ Crear gráfica de barras dinámica
    fig_capacidad = px.bar(
        capacidad_servicio,
        x="SERVICIO",
        y="CAPACIDAD_PASAJEROS",
        title="🧍‍♂️ Capacidad Promedio de Pasajeros por Tipo de Servicio",
        labels={
            "SERVICIO": "Tipo de Servicio",
            "CAPACIDAD_PASAJEROS": "Promedio de Pasajeros",
        },
        text="CAPACIDAD_PASAJEROS",
        color="SERVICIO",
        color_discrete_sequence=px.colors.qualitative.Set3  # 🎨 paleta consistente con tus otras gráficas
    )

    # ✅ Personalización visual
    fig_capacidad.update_traces(
        texttemplate="%{text:.1f}",
        textposition="outside",
        marker=dict(line=dict(color="black", width=1))
    )

    fig_capacidad.update_layout(
        height=500,
        xaxis_tickangle=-30,
        showlegend=False,
        plot_bgcolor="white"
    )

    fig_capacidad.show()

else:
    print("❌ No hay datos disponibles o faltan columnas necesarias ('CAPACIDAD_PASAJEROS', 'SERVICIO') para crear la gráfica.")


In [ ]:
# 🌎 Gráfica dinámica: Top 3 departamentos con más vehículos
import plotly.express as px

if df is not None and not df.empty and "DEPARTAMENTO" in df.columns:
    # ✅ Contar vehículos por departamento
    departamento_counts = df["DEPARTAMENTO"].value_counts().reset_index()
    departamento_counts.columns = ["Departamento", "Cantidad"]

    # ✅ Separar el Top 3
    top3 = departamento_counts.head(3)

    print("🌎 Creando gráfica del Top 3 departamentos con más vehículos...")

    # ✅ Crear gráfica de barras dinámica
    fig_top3 = px.bar(
        top3,
        x="Departamento",
        y="Cantidad",
        title="🌎 Top 3 Departamentos con Más Vehículos",
        labels={"Departamento": "Departamento", "Cantidad": "Cantidad de Vehículos"},
        text="Cantidad",
        color="Departamento",
        color_discrete_sequence=px.colors.qualitative.G10
    )

    # ✅ Personalización visual
    fig_top3.update_traces(
        texttemplate="%{text}",
        textposition="outside",
        marker=dict(line=dict(color="black", width=1))
    )

    fig_top3.update_layout(
        height=500,
        yaxis_range=[0, top3["Cantidad"].max() * 1.2],  # Espacio extra arriba
        xaxis_tickangle=-15,
        showlegend=False,
        plot_bgcolor="white"
    )

    fig_top3.show()

else:
    print("❌ No hay datos disponibles o falta la columna 'DEPARTAMENTO' para crear la gráfica.")


In [ ]:
if df is not None and not df.empty:
    # Contar la cantidad de vehículos por tipo de servicio
    servicios = (
        df["SERVICIO"]
        .value_counts()
        .reset_index()
    )
    servicios.columns = ["Servicio", "Cantidad"]

    print("⚡ Creando gráfica de cantidad de vehículos eléctricos por tipo de servicio...")

    # Crear gráfico de barras horizontal
    fig_servicio = px.bar(
        servicios,
        x="Cantidad",
        y="Servicio",
        orientation="h",
        text="Cantidad",  # ✅ Mostrar los valores sobre las barras
        title="⚡ Cantidad de Vehículos Eléctricos por Tipo de Servicio",
        labels={"Cantidad": "Cantidad de vehículos", "Servicio": "Tipo de servicio"},
        color="Servicio",  # ✅ Colores distintos para cada categoría
        color_discrete_sequence=px.colors.qualitative.Vivid
    )

    # Personalización de trazos
    fig_servicio.update_traces(
        textposition="outside",
        marker=dict(line=dict(color="black", width=1))  # ✅ Borde negro
    )

    # Ajustes de diseño y estilo
    fig_servicio.update_layout(
        height=500,
        xaxis_title="Cantidad de vehículos",
        yaxis_title="Tipo de servicio",
        yaxis=dict(categoryorder="total ascending"),  # ✅ Orden ascendente
        plot_bgcolor="white",
        showlegend=False,
        title_font=dict(size=22),
        font=dict(size=14)
    )

    fig_servicio.show()

## USO_ESTACIONES

In [ ]:
# Ruta del archivo
ruta = "./data/uso_estaciones.csv"
df = leer_archivo(ruta)

In [ ]:
# Convertir columnas de consumo a numéricas, manejando errores
df["TOTAL DE CONSUMO"] = pd.to_numeric(df["TOTAL DE CONSUMO"], errors="coerce")
# Eliminar columnas 'Consumo Primera Mitad' y 'Consumo Segunda Mitad' que ya no son necesarias
df = df.drop(columns=["CONSUMO PRIMERA MITAD DEL MES", "CONSUMO SEGUNDA MITAD DEL MES"])
mostrar_datos(df, 10)

In [ ]:
# ---------------------------
# 6. Consumo total mensual (versión Plotly Express)
# ---------------------------
import plotly.express as px

if "MES" in df.columns and "AÑO" in df.columns and "TOTAL DE CONSUMO" in df.columns:
    # Combinar mes y año en una sola columna para el eje X
    df["Periodo"] = df["MES"] + " " + df["AÑO"].astype(str)

    print("📊 Creando gráfica de consumo total mensual...")

    fig_consumo = px.line(
        df,
        x="Periodo",
        y="TOTAL DE CONSUMO",
        title="📈 Consumo Total Mensual",
        labels={
            "Periodo": "Mes y Año",
            "TOTAL DE CONSUMO": "Consumo Total"
        },
        markers=True,  # ✅ Muestra puntos sobre la línea
        line_shape="linear",
        color_discrete_sequence=px.colors.qualitative.Set3
    )

    # Personalizar trazas y etiquetas
    fig_consumo.update_traces(
        text=df["TOTAL DE CONSUMO"].apply(lambda x: f"{int(x):,}"),
        textposition="top center",
        hovertemplate="<b>%{x}</b><br>Consumo: %{y:,}"
    )

    # Ajustes visuales del layout
    fig_consumo.update_layout(
        height=500,
        xaxis_tickangle=-45,
        font=dict(size=12),
        plot_bgcolor="white",
        hovermode="x unified",
        margin=dict(l=40, r=40, t=80, b=80)
    )

    # Líneas de cuadrícula suaves
    fig_consumo.update_xaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgray")
    fig_consumo.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgray", tickformat=",d")

    fig_consumo.show()

else:
    print("❌ No se encontraron las columnas necesarias ('MES', 'AÑO', 'TOTAL DE CONSUMO').")


In [ ]:
# ---------------------------
# 7. Consumo Promedio Anual (versión Plotly Express)
# ---------------------------
import plotly.express as px

if "AÑO" in df.columns and "TOTAL DE CONSUMO" in df.columns:
    # Asegurar tipo string para el eje X
    df["AÑO"] = df["AÑO"].astype(str)

    # Calcular consumo promedio anual
    consumo_anual = df.groupby("AÑO", as_index=False)["TOTAL DE CONSUMO"].mean()

    print("📊 Creando gráfica de consumo promedio anual...")

    fig_consumo_anual = px.bar(
        consumo_anual,
        x="AÑO",
        y="TOTAL DE CONSUMO",
        title="📊 Consumo Promedio Anual",
        labels={
            "AÑO": "Año",
            "TOTAL DE CONSUMO": "Consumo Promedio"
        },
        text="TOTAL DE CONSUMO",
        color="AÑO",
        color_discrete_sequence=px.colors.qualitative.Set3
    )

    # Personalizar etiquetas y formato
    fig_consumo_anual.update_traces(
        texttemplate="%{y:,.0f}",
        textposition="outside",
        marker_line_color="black",
        marker_line_width=1
    )

    # Ajustes visuales del layout
    fig_consumo_anual.update_layout(
        height=500,
        xaxis_tickangle=-30,
        font=dict(size=12),
        plot_bgcolor="white",
        showlegend=False,
        margin=dict(l=40, r=40, t=80, b=60)
    )

    # Líneas de cuadrícula suaves
    fig_consumo_anual.update_xaxes(showgrid=False)
    fig_consumo_anual.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgray", tickformat=",d")

    fig_consumo_anual.show()

else:
    print("❌ No se encontraron las columnas necesarias ('AÑO', 'TOTAL DE CONSUMO').")
